# Extraction de l'âge probable du compte reviewer

Ce notebook permet d'estimer la date de création du compte (et son âge) pour chaque reviewer présent dans le fichier `reviews_select.csv`.

### Méthodologie
Sur Airbnb, les identifiants d'utilisateurs (`host_id` et `reviewer_id`) proviennent généralement de la même séquence numérique auto-incrémentée.
En sachant quand un hôte a créé son compte grâce à la colonne `host_since` de `listings.csv`, nous pouvons **interpoler la date de création** d'un compte reviewer en fonction de son `reviewer_id`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Options d'affichage
pd.set_option('display.max_columns', None)

## 1. Chargement des données de référence (Hôtes)

Nous utilisons `listings.csv` pour extraire la correspondance entre un ID utilisateur (`host_id`) et sa date d'inscription (`host_since`).

In [ ]:
print("Chargement des dates de création des comptes hôtes...")
df_listings = pd.read_csv('./data/listings.csv', usecols=['host_id', 'host_since'])

# Suppression des doublons et des valeurs manquantes
df_hosts = df_listings.drop_duplicates(subset=['host_id']).dropna()
df_hosts['host_since'] = pd.to_datetime(df_hosts['host_since'])

# Tri par host_id pour s'assurer que la séquence est croissante (nécessaire pour l'interpolation)
df_hosts = df_hosts.sort_values('host_id').reset_index(drop=True)

# Extraction des tableaux numpy pour l'interpolation de scipy/numpy
reference_ids = df_hosts['host_id'].values
# Conversion des dates en format numérique (nanosecondes)
reference_dates = df_hosts['host_since'].astype('int64').values

print(f"{len(reference_ids)} hôtes uniques utilisés comme référence.")

## 2. Chargement des données des reviewers

Nous chargeons ensuite le fichier cible `reviews_select.csv`.

In [ ]:
print("Chargement du fichier reviews_select.csv...")
df_reviews = pd.read_csv('./data/reviews_select.csv')
df_reviews['date'] = pd.to_datetime(df_reviews['date'])

reviewer_ids = df_reviews['reviewer_id'].values
print(f"Nombre de commentaires chargés : {len(reviewer_ids)}")

## 3. Interpolation des dates de création des comptes Reviewers

Pour chaque `reviewer_id`, nous interpolons sa date probable de création de compte en fonction des IDs hôtes environnants.

In [ ]:
print("Interpolation des dates de création probables...")
# numpy.interp permet l'interpolation linéaire 1D
estimated_dates_numeric = np.interp(reviewer_ids, reference_ids, reference_dates)

# Reconversion en format datetime
df_reviews['estimated_account_created'] = pd.to_datetime(estimated_dates_numeric)

# Calcul de l'âge du compte au moment du commentaire (en jours puis années)
df_reviews['account_age_at_review_days'] = (df_reviews['date'] - df_reviews['estimated_account_created']).dt.days

# On borne à 0 minimum (pour de rares anomalies d'interpolation)
df_reviews['account_age_at_review_days'] = df_reviews['account_age_at_review_days'].clip(lower=0)
df_reviews['account_age_at_review_years'] = df_reviews['account_age_at_review_days'] / 365.25

print("Aperçu des résultats :")
df_reviews[['reviewer_id', 'date', 'estimated_account_created', 'account_age_at_review_years']].head(10)

## 4. Visualisation

Regardons la distribution de l'âge des comptes (au moment où le commentaire a été posté).

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_reviews['account_age_at_review_years'].dropna(), bins=30, kde=True, color='skyblue')
plt.title("Distribution de l'âge probable des comptes (en années)")
plt.xlabel("Âge du compte au moment du commentaire (Années)")
plt.ylabel("Nombre de commentaires")
plt.grid(axis='y', alpha=0.7)
plt.show()

## 5. Sauvegarde des résultats

On exporte les données enrichies dans un nouveau fichier CSV `reviews_select_with_age.csv`.

In [ ]:
output_file = './data/reviews_select_with_age.csv'
df_reviews.to_csv(output_file, index=False)
print(f"Fichier enrichi sauvegardé avec succès dans: {output_file}")
df_reviews.head()